#### 1、什么是Tool
Tool（工具）是智能体可以调用的外部函数，让AI能够执行特定任务，如查询天气、搜索信息、计算数据等。

**关键特征**：

- 有明确的输入参数
- 执行具体功能
- 返回结构化结果
- 可以被多个智能体复用



 #### 2、为什么要用Tool

1. **扩展能力**：弥补AI的局限性
2. **获取实时数据**：访问最新信息
3. **执行操作**：调用外部系统API
4. **确保准确性**：减少AI幻觉
5. **标准化流程**：统一业务逻辑

#### 3、如何使用Tool

使用**Responses API** 携带工具

In [1]:
from openai import OpenAI
from  dotenv import load_dotenv
import os

# 加载环境变量
load_dotenv()

# 创建客户端 - 需要传入api_key和base_url
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

# API调用（创建响应）
# response = client.responses.create(
#     model=os.getenv("OPENAI_MODEL_NAME"),
#     tools=[{"type": "web_search"}],
#     input="今天有什么积极的新闻吗?"
# )
# print(response.output_text)


client.chat.completions.create(
    # model=os.getenv("SF_MODEL_NAME"),
    model=os.getenv("OPENAI_MODEL_NAME"),
    messages=[
        {"role": "system", "content": "你是一个搜索专家"},
        {
            "role": "user",
            "content": "今天有什么积极的新闻吗？",
        },
    ],
    tools=[{"type": "web_search"}],
)

# output---message类型的output_text类型的输出项

BadRequestError: Error code: 400 - {'code': 20015, 'message': "Input should be 'function'", 'data': None}

这里注意，传统 **Chat Completions API使用不能这样使用**。它的type必须是funcation `“type": "funcation"`不能是工具的名字。

官网链接：https://platform.openai.com/docs/guides/tools?tool-type=function-calling

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

# 加载环境变量
load_dotenv()

# 创建客户端
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)


# 定义工具函数
def get_weather(city: str) -> str:
    """模拟查询天气函数"""
    weather_data = {
        "北京": "北京：晴天，15-25°C，适宜外出",
        "上海": "上海：多云，18-28°C，微风",
        "广州": "广州：阵雨，22-30°C，记得带伞",
        "深圳": "深圳：晴转多云，23-31°C，湿度较高"
    }

    return weather_data.get(city, f"暂无{city}的天气信息")


# 使用底层JSON方式定义工具
def chat_with_basic_tools():
    print("=== 1. 底层方式：手动编写工具定义 ===")

    messages = [
        {"role": "system", "content": "你是一个天气助手，可以查询天气信息。"},
        {"role": "user", "content": "我想知道北京的天气怎么样？"}
    ]

    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME"),
        messages=messages,
        tools=[{
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "查询指定城市的天气信息",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {
                            "type": "string",
                            "description": "城市名称，如'北京'、'上海'",
                        },
                    },
                    "required": ["city"],
                    "additionalProperties": False
                },
            }
        }],
    )
    print(response)

    message = response.choices[0].message
    messages.append(message)
    #
    if message.tool_calls:
        for tool_call in message.tool_calls:
            if tool_call.function.name == "get_weather":
                args = json.loads(tool_call.function.arguments)
                result = get_weather(args["city"]) # 调用工具
                print(result)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })

        second_response = client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL_NAME"),
            messages=messages,
        )
        print(f"最终回复: {second_response.choices[0].message.content}")

    return response


if __name__ == "__main__":
    chat_with_basic_tools()


=== 1. 底层方式：手动编写工具定义 ===
ChatCompletion(id='019dd9f8d99db63d2b10ade3081ae261', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='019dd9f8f3bf424642313da12c06477b', function=Function(arguments='{"city": "北京"}', name='get_weather'), type='function', index=0)], reasoning_content='\n好的，用户想知道北京的天气。我需要调用get_weather这个工具来获取信息。首先，确认用户提供的城市名称是“北京”，参数正确。然后检查工具的函数签名，确保参数是city且类型为字符串。没有其他参数需要处理，所以构造一个包含city: "北京"的JSON对象。最后，生成符合要求的tool_call结构。确保没有遗漏必要字段，比如函数名和参数。现在可以正确返回工具调用了。\n'))], created=1777478457, model='Qwen/Qwen3-32B', object='chat.completion', service_tier=None, system_fingerprint='', usage=CompletionUsage(completion_tokens=122, prompt_tokens=188, total_tokens=310, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=102, rejected_pr

In [23]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

def get_weather(city: str) -> str:
    """模拟查询天气函数"""
    weather_data = {
        "北京": "北京：晴天，15-25°C，适宜外出",
        "上海": "上海：多云，18-28°C，微风",
        "广州": "广州：阵雨，22-30°C，记得带伞",
        "深圳": "深圳：晴转多云，23-31°C，湿度较高"
    }
    print("xxxxxxxxxxx")
    return weather_data.get(city, f"暂无{city}的天气信息")



tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country e.g. Bogotá, Colombia",
                }
            },
            "required": ["location"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

response = client.responses.create(
     model=os.getenv("OPENAI_MODEL_NAME"),
    tools=[{"type": "web_search"}],
    input="What was a positive news story from today?"
)

print(response)

Response(id='resp_0104e992d022c1060069f41f53c5888197a7a817aed61b0bd8', created_at=None, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0104e992d022c1060069f41f5481748197b90c51378774ffde', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionWebSearch(id='ws_0104e992d022c1060069f41f5659988197979a3141c0edd48e', action=None, status='completed', type='web_search_call'), ResponseReasoningItem(id='rs_0104e992d022c1060069f41f5888bc8197b0fa49c107626f95', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionWebSearch(id='ws_0104e992d022c1060069f41f5972d081978c4de9a7a65caf04', action=None, status='completed', type='web_search_call'), ResponseReasoningItem(id='rs_0104e992d022c1060069f41f5b66e48197b52eacd0d59a8941', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), Respo

#### 4、Tool使用场景

| 场景     | 工具示例                              | 说明           |
| :------- | :------------------------------------ | :------------- |
| 数据查询 | `search_product`, `query_order`       | 查询数据库信息 |
| 计算服务 | `calculate_price`, `convert_currency` | 执行数学计算   |
| 外部API  | `send_email`, `create_ticket`         | 调用第三方服务 |
| 文件操作 | `read_file`, `generate_report`        | 处理文档数据   |

- Response API 在工具处理上属于半自动化(在线工具上)---找工具--->调用工具（自定义的工具）== Chat completions（等价的）
- Chat completions 在工具处理上属于手动（自定调用工具）模型只找到工具。